# Enzymatic Reaction Example

This notebook simulates a simple enzymatic (Michaelis-Menten) reaction and generates publication-quality plots.

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from scipy.integrate import odeint

In [ ]:
# --- Publication-quality plot settings ---

# Fix Type 3 font compliance issue: use TrueType (Type 42) fonts in PDF/PS output
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

# Use matplotlib's built-in mathtext (mathtex) renderer for math labels
matplotlib.rcParams['text.usetex'] = False  # Use matplotlib mathtext (no external LaTeX required)
matplotlib.rcParams['mathtext.fontset'] = 'cm'  # Computer Modern for publication-quality math

# Font sizes
SMALL_SIZE  = 14
MEDIUM_SIZE = 16
LARGE_SIZE  = 18

matplotlib.rcParams['font.size']        = MEDIUM_SIZE   # default text size
matplotlib.rcParams['axes.titlesize']   = LARGE_SIZE    # axes title
matplotlib.rcParams['axes.labelsize']   = LARGE_SIZE    # x and y axis label size
matplotlib.rcParams['xtick.labelsize']  = MEDIUM_SIZE   # x tick label size
matplotlib.rcParams['ytick.labelsize']  = MEDIUM_SIZE   # y tick label size
matplotlib.rcParams['legend.fontsize']  = SMALL_SIZE    # legend font size
matplotlib.rcParams['figure.titlesize'] = LARGE_SIZE    # figure title size

# Tick parameters
matplotlib.rcParams['xtick.major.size']  = 7
matplotlib.rcParams['xtick.major.width'] = 1.5
matplotlib.rcParams['ytick.major.size']  = 7
matplotlib.rcParams['ytick.major.width'] = 1.5
matplotlib.rcParams['xtick.minor.size']  = 4
matplotlib.rcParams['xtick.minor.width'] = 1.0
matplotlib.rcParams['ytick.minor.size']  = 4
matplotlib.rcParams['ytick.minor.width'] = 1.0

# Line widths
matplotlib.rcParams['lines.linewidth']  = 2.0
matplotlib.rcParams['axes.linewidth']   = 1.5

print('Matplotlib rcParams configured for publication-quality output.')

In [ ]:
# --- Michaelis-Menten enzymatic reaction model ---
#
# Reaction scheme:
#   E + S <--(k_r)--> ES --(k_cat)--> E + P
#          --(k_f)-->
#
# Species: S (substrate), E (enzyme), ES (enzyme-substrate complex), P (product)

# Rate constants
k_f   = 1.0   # forward binding rate  (uM^-1 s^-1)
k_r   = 0.1   # reverse unbinding rate (s^-1)
k_cat = 0.5   # catalytic rate         (s^-1)

# Initial conditions (in uM)
S0  = 10.0   # initial substrate
E0  =  1.0   # total enzyme
ES0 =  0.0   # enzyme-substrate complex
P0  =  0.0   # initial product

y0 = [S0, E0, ES0, P0]

def enzymatic_odes(y, t, k_f, k_r, k_cat):
    S, E, ES, P = y
    dS  = -k_f * E * S + k_r * ES
    dE  = -k_f * E * S + k_r * ES + k_cat * ES
    dES =  k_f * E * S - k_r * ES - k_cat * ES
    dP  =  k_cat * ES
    return [dS, dE, dES, dP]

# Time vector
t = np.linspace(0, 30, 500)

# Solve ODE
sol = odeint(enzymatic_odes, y0, t, args=(k_f, k_r, k_cat))
S, E, ES, P = sol.T

print('ODE integration complete.')

In [ ]:
# --- Figure 1: Species concentrations over time ---

fig, ax = plt.subplots(figsize=(7, 5))

ax.plot(t, S,  label=r'$[S]$  (substrate)')
ax.plot(t, E,  label=r'$[E]$  (free enzyme)')
ax.plot(t, ES, label=r'$[ES]$ (complex)')
ax.plot(t, P,  label=r'$[P]$  (product)')

# Axis labels in math mode
ax.set_xlabel(r'Time $t$ (s)')
ax.set_ylabel(r'Concentration ($\mu$M)')
ax.set_title(r'Michaelis-Menten Kinetics')

# Tick parameters: size and label size set globally via rcParams above;
# apply minor ticks and direction here
ax.tick_params(which='both', direction='in', top=True, right=True)
ax.tick_params(which='major', labelsize=matplotlib.rcParams['xtick.labelsize'])
ax.minorticks_on()

ax.legend()
fig.tight_layout()
plt.savefig('enzymatic_kinetics.pdf', bbox_inches='tight')
plt.show()
print('Figure 1 saved to enzymatic_kinetics.pdf')

In [ ]:
# --- Figure 2: Michaelis-Menten velocity curve (v vs [S]) ---
#
# v_max = k_cat * E_total
# K_M   = (k_r + k_cat) / k_f
# v     = v_max * [S] / (K_M + [S])

E_total = E0
v_max   = k_cat * E_total
K_M     = (k_r + k_cat) / k_f

S_range = np.linspace(0, 30, 300)
v       = v_max * S_range / (K_M + S_range)

fig, ax = plt.subplots(figsize=(7, 5))

ax.plot(S_range, v, color='steelblue')
ax.axhline(v_max, color='gray',      linestyle='--', linewidth=1.5,
           label=r'$V_{\mathrm{max}} = k_{\mathrm{cat}}[E]_0$')
ax.axhline(v_max / 2, color='salmon', linestyle=':',  linewidth=1.5,
           label=r'$V_{\mathrm{max}}/2$')
ax.axvline(K_M,       color='salmon', linestyle=':',  linewidth=1.5,
           label=r'$K_M = (k_r + k_{\mathrm{cat}})/k_f$')

# Axis labels in math mode
ax.set_xlabel(r'Substrate concentration $[S]$ ($\mu$M)')
ax.set_ylabel(r'Reaction velocity $v$ ($\mu$M s$^{-1}$)')
ax.set_title(r'Michaelis-Menten Velocity Curve')

ax.tick_params(which='both', direction='in', top=True, right=True)
ax.tick_params(which='major', labelsize=matplotlib.rcParams['xtick.labelsize'])
ax.minorticks_on()

ax.legend()
fig.tight_layout()
plt.savefig('michaelis_menten_velocity.pdf', bbox_inches='tight')
plt.show()
print('Figure 2 saved to michaelis_menten_velocity.pdf')